# NVD Analysis Backlog

On February 15, 2024, NVD significantly reduced its CVE analysis capacity. This
notebook tracks the resulting backlog: how many CVEs are awaiting analysis, how
fast the gap is growing, and how long it would take to clear at historical rates.

In [ ]:
from datetime import datetime
from vulnflow.helpers import CacheBuilder

CACHE_PATH = "../data/complete_library.parquet"

ds = CacheBuilder().ensure(CACHE_PATH)

# Include all statuses (especially rejected and awaiting)
df = ds.to_pandas(start="2024-02-15", rejected=True)
days_since = (datetime.now() - datetime(2024, 2, 15)).days

print(f"{len(df):,} CVEs since Feb 15, 2024 ({days_since} days)")

## Status Distribution

In [ ]:
import matplotlib.pyplot as plt

STATUS_COLORS = {
    "analyzed": "#2ecc71",
    "awaiting": "#e74c3c",
    "modified": "#f39c12",
    "rejected": "#95a5a6",
    "deferred": "#9b59b6",
}

status_counts = df["nvd_status"].value_counts()
total = status_counts.sum()
colors = [STATUS_COLORS.get(s, "#cccccc") for s in status_counts.index]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(status_counts.index[::-1], status_counts.values[::-1],
               color=colors[::-1], edgecolor="white")
for bar, val in zip(bars, status_counts.values[::-1]):
    pct = 100 * val / total
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height() / 2,
            f"{val:,} ({pct:.1f}%)", va="center", fontsize=12, fontweight="bold")
ax.set_title(f"NVD Status Distribution (since Feb 15, 2024)", fontsize=16)
ax.set_xlabel("CVE Count", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5, axis="x")
plt.tight_layout()
plt.show()

## Backlog Estimate

Estimate how long it would take to clear the backlog at NVD's pre-Feb-2024
analysis rate (~55 CVEs/day).

In [ ]:
backlog_statuses = ["awaiting", "modified"]
backlog_count = status_counts.reindex(backlog_statuses).fillna(0).sum()
analyzed_count = status_counts.get("analyzed", 0)

PRE_FEB_RATE = 55  # CVEs analyzed per day, historical average
days_to_clear = int(backlog_count / PRE_FEB_RATE)

print(f"Backlog (awaiting + modified): {int(backlog_count):,}")
print(f"Already analyzed: {analyzed_count:,}")
print(f"Analysis rate needed: {PRE_FEB_RATE} CVEs/day")
print(f"Estimated time to clear: {days_to_clear:,} days ({days_to_clear / 365:.1f} years)")
print(f"\nNote: new CVEs keep arriving (~80-100/day), so the real clearance time is longer.")

## Monthly Publish vs Analyze Rates

In [ ]:
import pandas as pd

df["pub_month"] = pd.to_datetime(df["nvd_published_at"]).dt.to_period("M")

# Published per month (all statuses)
published_monthly = df.groupby("pub_month").size().reset_index(name="published")

# Analyzed per month (only analyzed status)
analyzed_monthly = (
    df[df["nvd_status"] == "analyzed"]
    .groupby("pub_month").size()
    .reset_index(name="analyzed")
)

combined = published_monthly.merge(analyzed_monthly, on="pub_month", how="left").fillna(0)
combined["analyzed"] = combined["analyzed"].astype(int)

fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(combined))
ax.plot(x, combined["published"], marker="o", color="#1f77b4", linewidth=2,
        markersize=5, label="Published")
ax.plot(x, combined["analyzed"], marker="s", color="#2ecc71", linewidth=2,
        markersize=5, label="Analyzed")
ax.fill_between(x, combined["analyzed"], combined["published"],
                alpha=0.15, color="#e74c3c", label="Gap (backlog growth)")

tick_pos = list(range(0, len(combined), max(1, len(combined) // 10)))
tick_labels = [str(combined["pub_month"].iloc[i]) for i in tick_pos]
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, rotation=45, ha="right")

ax.set_title("Monthly Published vs Analyzed CVEs", fontsize=16)
ax.set_ylabel("CVE Count", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Cumulative Backlog Growth

In [ ]:
combined["cum_published"] = combined["published"].cumsum()
combined["cum_analyzed"] = combined["analyzed"].cumsum()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(x, combined["cum_published"], color="#1f77b4", linewidth=2, label="Total published")
ax.plot(x, combined["cum_analyzed"], color="#2ecc71", linewidth=2, label="Total analyzed")
ax.fill_between(x, combined["cum_analyzed"], combined["cum_published"],
                alpha=0.2, color="#e74c3c", label="Cumulative backlog")

# Annotate final gap
gap = int(combined["cum_published"].iloc[-1] - combined["cum_analyzed"].iloc[-1])
ax.annotate(
    f"Backlog: {gap:,}",
    xy=(len(combined) - 1, combined["cum_published"].iloc[-1]),
    xytext=(len(combined) - 4, combined["cum_published"].iloc[-1] * 0.7),
    arrowprops=dict(arrowstyle="->", color="#e74c3c"),
    fontsize=12, color="#e74c3c", fontweight="bold",
)

ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels, rotation=45, ha="right")

ax.set_title("Cumulative Published vs Analyzed CVEs", fontsize=16)
ax.set_ylabel("Cumulative Count", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()